# Dense MoE Basics with PyTorch

Muc tieu cell dau:
- hieu `dense MoE` = moi token di qua tat ca experts
- router chi dong vai tro cho trong so de tron output
- chua lam sparse routing hay top-k


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='mps')

In [2]:
class Expert(nn.Module):
    def __init__(self, dModel: int, dHidden: int):
        super().__init__()
        self.fc1 = nn.Linear(dModel, dHidden)
        self.fc2 = nn.Linear(dHidden, dModel)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = F.gelu(self.fc1(x))
        return self.fc2(hidden)


class DenseMoE(nn.Module):
    def __init__(self, dModel: int, dHidden: int, nExperts: int):
        super().__init__()
        self.experts = nn.ModuleList(
            [Expert(dModel=dModel, dHidden=dHidden) for _ in range(nExperts)]
        )
        self.router = nn.Linear(dModel, nExperts)

    def forward(self, x: torch.Tensor):
        routerLogits = self.router(x)
        gates = F.softmax(routerLogits, dim=-1)

        expertOutputs = []
        for expert in self.experts:
            expertOutputs.append(expert(x))

        expertOutputs = torch.stack(expertOutputs, dim=2)
        mixedOutput = torch.sum(expertOutputs * gates.unsqueeze(-1), dim=2)
        return mixedOutput, gates, expertOutputs


In [3]:
batchSize = 2
seqLen = 3
dModel = 8
dHidden = 16
nExperts = 4

x = torch.randn(batchSize, seqLen, dModel, device=device)
model = DenseMoE(dModel=dModel, dHidden=dHidden, nExperts=nExperts).to(device)

output, gates, expertOutputs = model(x)

print("input shape:", x.shape)
print("router gates shape:", gates.shape)
print("expert outputs shape:", expertOutputs.shape)
print("final output shape:", output.shape)

input shape: torch.Size([2, 3, 8])
router gates shape: torch.Size([2, 3, 4])
expert outputs shape: torch.Size([2, 3, 4, 8])
final output shape: torch.Size([2, 3, 8])


In [4]:
print("gates for token [0, 0]:")
print(gates[0, 0])
print("sum of gates for token [0, 0]:", gates[0, 0].sum())

print("\nfirst 2 dims of each expert output for token [0, 0]:")
for expertIdx in range(nExperts):
    print(f"expert {expertIdx}:", expertOutputs[0, 0, expertIdx, :2])

print("\nfinal mixed output first 2 dims:", output[0, 0, :2])

gates for token [0, 0]:
tensor([0.0423, 0.5734, 0.3391, 0.0452], device='mps:0',
       grad_fn=<SelectBackward0>)
sum of gates for token [0, 0]: tensor(1., device='mps:0', grad_fn=<SumBackward0>)

first 2 dims of each expert output for token [0, 0]:
expert 0: tensor([0.5296, 0.0916], device='mps:0', grad_fn=<SliceBackward0>)
expert 1: tensor([0.1327, 0.1053], device='mps:0', grad_fn=<SliceBackward0>)
expert 2: tensor([-0.0716, -0.4176], device='mps:0', grad_fn=<SliceBackward0>)
expert 3: tensor([-0.5948, -0.1637], device='mps:0', grad_fn=<SliceBackward0>)

final mixed output first 2 dims: tensor([ 0.0474, -0.0847], device='mps:0', grad_fn=<SliceBackward0>)


## Doc ket qua

- `gates.shape = (batch, seq, nExperts)`
- `expertOutputs.shape = (batch, seq, nExperts, dModel)`
- `output.shape = (batch, seq, dModel)`

Y nghia:
- moi token deu di qua tat ca experts
- router chi cho trong so de tron
- day la `dense MoE`

Buoc tiep theo sau khi ban on phan nay:
- doi `DenseMoE` thanh `SparseMoE`
- thay soft weighted all-experts bang `top-k routing`
